# Aeon CNN for Time Series Classification

This notebook demonstrates how to use the Aeon CNN model for time series classification.

In [ ]:
import os
import sys
import time
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# Add the parent directory to the path to import from src
sys.path.append("..")

# Set random seed for reproducibility
np.random.seed(42)

## 1. Load and Prepare Data

In [ ]:
def load_npy_dataset(file_path):
    """
    Load and prepare numpy dataset for time series classification.

    Args:
        file_path: Path to the .npy file containing the dataset

    Returns:
        Tuple containing (X_train, y_train, X_test, y_test)
    """
    data = np.load(file_path, allow_pickle=True).item()

    X_train = data["train"]["X"]
    y_train = np.array([int(x) for x in data["train"]["y"]])
    X_test = data["test"]["X"]
    y_test = np.array([int(x) for x in data["test"]["y"]])

    print(f"X_train shape: {X_train.shape}")
    print(f"X_test shape: {X_test.shape}")
    print(f"y_train shape: {y_train.shape}")
    print(f"y_test shape: {y_test.shape}")

    return X_train, y_train, X_test, y_test

In [ ]:
# Specify the path to your dataset
dataset_path = "../data/raw/your_dataset.npy"  # Change this to your dataset path

# Load the dataset
X_train, y_train, X_test, y_test = load_npy_dataset(dataset_path)

## 2. Visualize Some Examples

In [ ]:
def plot_time_series_examples(X, y, n_examples=3):
    """Plot a few examples of time series from each class."""
    classes = np.unique(y)
    n_classes = len(classes)

    plt.figure(figsize=(15, n_classes * 3))

    for i, cls in enumerate(classes):
        # Get indices of examples from this class
        idx = np.where(y == cls)[0][:n_examples]

        for j, example_idx in enumerate(idx):
            plt.subplot(n_classes, n_examples, i * n_examples + j + 1)

            if len(X.shape) == 3:  # Multivariate
                for dim in range(X.shape[2]):
                    plt.plot(X[example_idx, :, dim], label=f"Dim {dim}")
                if X.shape[2] > 1:
                    plt.legend(loc="upper right")
            else:  # Univariate
                plt.plot(X[example_idx])

            plt.title(f"Class {cls}")
            plt.grid(True)

    plt.tight_layout()
    plt.show()

In [ ]:
# Visualize examples
plot_time_series_examples(X_train, y_train)

## 3. Import and Configure Aeon CNN Model

In [ ]:
from aeon.classification.deep_learning import CNNClassifier

# Alternative: Import our wrapper class
# from src.models.aeon_cnn.model import AeonCNNModel

In [ ]:
# Initialize the CNNClassifier with desired parameters
clf = CNNClassifier(
    n_epochs=500,  # Number of training epochs
    batch_size=16,  # Batch size
    kernel_size=7,  # Size of convolutional kernel
    n_filters=16,  # Number of convolutional filters
    random_state=42,  # For reproducibility
)

## 4. Train the Model

In [ ]:
# Record the start time
start_time = time.time()

# Fit the model
clf.fit(X_train, y_train)

# Calculate training time
fit_time = time.time() - start_time
print(f"Training completed in {fit_time:.2f} seconds")

## 5. Evaluate the Model

In [ ]:
# Measure prediction time for training data
start_pred_train_time = time.time()
y_pred_train = clf.predict(X_train)
pred_train_time = time.time() - start_pred_train_time

# Measure prediction time for test data
start_pred_test_time = time.time()
y_pred_test = clf.predict(X_test)
pred_test_time = time.time() - start_pred_test_time

# Calculate accuracy
train_accuracy = accuracy_score(y_train, y_pred_train)
test_accuracy = accuracy_score(y_test, y_pred_test)

print(f"Training Accuracy: {train_accuracy:.4f}")
print(f"Test Accuracy: {test_accuracy:.4f}")
print(f"Training Prediction Time: {pred_train_time:.4f} seconds")
print(f"Test Prediction Time: {pred_test_time:.4f} seconds")

In [ ]:
# Calculate samples per second
train_samples_per_second = X_train.shape[0] / pred_train_time
test_samples_per_second = X_test.shape[0] / pred_test_time

print(f"Training Samples per Second: {train_samples_per_second:.2f}")
print(f"Test Samples per Second: {test_samples_per_second:.2f}")

In [ ]:
# Print classification report
print("Classification Report:")
print(classification_report(y_test, y_pred_test))

## 6. Visualize Confusion Matrix

In [ ]:
import seaborn as sns

# Calculate confusion matrix
cm = confusion_matrix(y_test, y_pred_test)
classes = np.unique(y_test)

plt.figure(figsize=(10, 8))
sns.heatmap(
    cm, annot=True, fmt="d", cmap="Blues", xticklabels=classes, yticklabels=classes
)
plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.title("Confusion Matrix")
plt.show()

## 7. Save Model and Results

In [ ]:
import pickle
import json
from datetime import datetime

# Create results directory if it doesn't exist
os.makedirs("../results", exist_ok=True)

# Save the model
with open(
    f"../results/aeon_cnn_model_{datetime.now().strftime('%Y%m%d_%H%M%S')}.pkl", "wb"
) as f:
    pickle.dump(clf, f)

# Compile results
results = {
    "model_name": "AeonCNN",
    "train_accuracy": float(train_accuracy),
    "test_accuracy": float(test_accuracy),
    "fit_time": float(fit_time),
    "pred_train_time": float(pred_train_time),
    "pred_test_time": float(pred_test_time),
    "train_samples_per_second": float(train_samples_per_second),
    "test_samples_per_second": float(test_samples_per_second),
    "dataset_name": os.path.basename(dataset_path).split(".")[0],
    "timestamp": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
}

# Save results as JSON
with open(
    f"../results/aeon_cnn_results_{datetime.now().strftime('%Y%m%d_%H%M%S')}.json", "w"
) as f:
    json.dump(results, f, indent=4)

## 8. Extended Analysis (Optional)

In [ ]:
# Get model size in bytes
def get_model_size(model):
    """Estimate the size of the model in bytes"""
    import pickle
    import sys

    return sys.getsizeof(pickle.dumps(model))


model_size_bytes = get_model_size(clf)
print(f"Model Size: {model_size_bytes / 1024:.2f} KB")

In [ ]:
# Analyze incorrectly classified instances
incorrect_indices = np.where(y_pred_test != y_test)[0]
print(f"Number of misclassified instances: {len(incorrect_indices)}")

if len(incorrect_indices) > 0:
    # Display a few misclassified instances
    n_examples = min(3, len(incorrect_indices))
    plt.figure(figsize=(15, n_examples * 3))

    for i in range(n_examples):
        idx = incorrect_indices[i]
        plt.subplot(n_examples, 1, i + 1)

        if len(X_test.shape) == 3:  # Multivariate
            for dim in range(X_test.shape[2]):
                plt.plot(X_test[idx, :, dim], label=f"Dim {dim}")
            if X_test.shape[2] > 1:
                plt.legend(loc="upper right")
        else:  # Univariate
            plt.plot(X_test[idx])

        plt.title(f"True: {y_test[idx]}, Predicted: {y_pred_test[idx]}")
        plt.grid(True)

    plt.tight_layout()
    plt.show()